**Practica de Grupo 9 - De acuerdo a las recomendaciones se busca otra pagina web a la del ejemplo y se ejecuta la actividad.**
**http://books.toscrape.com**

1.- Esta sección que instala los paquetes a utilizar



In [1]:
%pip -q install requests beautifulsoup4 lxml

Sección para importar los paquetes
1. requests, me permite hacer la solicitud HTTP.
2. BeautifulSoup, me ayuda a convertir el HTML en un árbol para poder recorrer.
3. time sirve para pausar el scraping y no sobrecargar la página.
4. urljoin, sirve para construir enlaces completos cuando en la web aparecen enlaces relativos.

In [3]:
import time, math, re
import requests
from urllib.parse import urljoin
from bs4 import BeautifulSoup

Aquí aprendí cómo descargar el HTML de una página web.
Uso requests.get(URL) para pedirle al servidor la página, y luego con .text obtengo el código HTML en formato de texto.

In [4]:
BASE = "http://books.toscrape.com/"
headers = {"User-Agent": "Mozilla/5.0"}

resp = requests.get(BASE, headers=headers, timeout=20)
print("status:", resp.status_code, "| bytes:", len(resp.text))
src = resp.text

status: 200 | bytes: 51294


El HTML descargado es solo texto plano. Con BeautifulSoup lo transformamos en un objeto llamado soup, que nos permite navegar fácilmente por etiquetas, atributos y clases.

In [5]:
soup = BeautifulSoup(src, "lxml")
print(soup.title.text)  # título de la página


    All products | Books to Scrape - Sandbox



Una de las partes más importantes que aprendí es cómo buscar etiquetas dentro del HTML, con .select("article.product_pod") busco todos los artículos de libros en la página.
Cada artículo tiene título, precio, rating y enlace dentro de distintas etiquetas.

In [6]:
# Por etiqueta + clase
pods = soup.select("article.product_pod")
len(pods), pods[0]

(20,
 <article class="product_pod">
 <div class="image_container">
 <a href="catalogue/a-light-in-the-attic_1000/index.html"><img alt="A Light in the Attic" class="thumbnail" src="media/cache/2c/da/2cdad67c44b002e7ead0cc35693c0e8b.jpg"/></a>
 </div>
 <p class="star-rating Three">
 <i class="icon-star"></i>
 <i class="icon-star"></i>
 <i class="icon-star"></i>
 <i class="icon-star"></i>
 <i class="icon-star"></i>
 </p>
 <h3><a href="catalogue/a-light-in-the-attic_1000/index.html" title="A Light in the Attic">A Light in the ...</a></h3>
 <div class="product_price">
 <p class="price_color">Â£51.77</p>
 <p class="instock availability">
 <i class="icon-ok"></i>
     
         In stock
     
 </p>
 <form>
 <button class="btn btn-primary btn-block" data-loading-text="Adding..." type="submit">Add to basket</button>
 </form>
 </div>
 </article>)

In [7]:
links = pods[0].select("h3 a")
links[0]

<a href="catalogue/a-light-in-the-attic_1000/index.html" title="A Light in the Attic">A Light in the ...</a>

In [8]:
soup.select("article.product_pod p.price_color")[:3]

[<p class="price_color">Â£51.77</p>,
 <p class="price_color">Â£53.74</p>,
 <p class="price_color">Â£50.10</p>]

In [9]:
soup.select("article.product_pod p.star-rating")[:3]

[<p class="star-rating Three">
 <i class="icon-star"></i>
 <i class="icon-star"></i>
 <i class="icon-star"></i>
 <i class="icon-star"></i>
 <i class="icon-star"></i>
 </p>,
 <p class="star-rating One">
 <i class="icon-star"></i>
 <i class="icon-star"></i>
 <i class="icon-star"></i>
 <i class="icon-star"></i>
 <i class="icon-star"></i>
 </p>,
 <p class="star-rating One">
 <i class="icon-star"></i>
 <i class="icon-star"></i>
 <i class="icon-star"></i>
 <i class="icon-star"></i>
 <i class="icon-star"></i>
 </p>]

Ya que encontré las etiquetas, aprendí a extraer la información que contienen:

tag.get_text(strip=True), con esta variante obtuve el texto dentro de una etiqueta.

tag["atributo"], accede a un atributo específico, como un enlace href.

También aprendí que algunas clases en HTML pueden servir para obtener valores (ej. rating).

In [10]:
pod = pods[0]

# título está en el atributo 'title' del <a>
title = pod.select_one("h3 a")["title"]

# precio en p.price_color (ej. '51.77')
price = pod.select_one("p.price_color").get_text(strip=True)

# rating: segunda clase de p.star-rating (ej. ['star-rating','Three'] → 'Three')
rating = pod.select_one("p.star-rating")["class"][1]

# enlace relativo → absoluto
href = pod.select_one("h3 a")["href"]
url  = urljoin(BASE, href)

title, price, rating, url

('A Light in the Attic',
 'Â£51.77',
 'Three',
 'http://books.toscrape.com/catalogue/a-light-in-the-attic_1000/index.html')

**Challenge 1 – Extraer todos los libros de la página**

Aquí practiqué cómo hacer un bucle for para recorrer todos los libros de la página y guardarlos como tuplas (título, precio, rating, url).

In [11]:
books = []
for pod in soup.select("article.product_pod"):
    title  = pod.select_one("h3 a")["title"]
    price  = pod.select_one("p.price_color").get_text(strip=True)
    rating = pod.select_one("p.star-rating")["class"][1]
    href   = pod.select_one("h3 a")["href"]
    url    = urljoin(BASE, href)
    books.append((title, price, rating, url))

len(books), books[:3]

(20,
 [('A Light in the Attic',
   'Â£51.77',
   'Three',
   'http://books.toscrape.com/catalogue/a-light-in-the-attic_1000/index.html'),
  ('Tipping the Velvet',
   'Â£53.74',
   'One',
   'http://books.toscrape.com/catalogue/tipping-the-velvet_999/index.html'),
  ('Soumission',
   'Â£50.10',
   'One',
   'http://books.toscrape.com/catalogue/soumission_998/index.html')])

**Challenge 2 – Función get_books**

Luego aprendí a modularizar el código, es decir, a meterlo dentro de una función para reutilizarlo en diferentes páginas.

In [12]:
def get_books(page_url: str):
    """Scrapea una página de listado de BooksToScrape y devuelve [(title, price, rating, url), ...]."""
    r = requests.get(page_url, headers=headers, timeout=20)
    s = BeautifulSoup(r.text, "lxml")
    out = []
    for pod in s.select("article.product_pod"):
        title  = pod.select_one("h3 a")["title"]
        price  = pod.select_one("p.price_color").get_text(strip=True)
        rating = pod.select_one("p.star-rating")["class"][1]
        href   = pod.select_one("h3 a")["href"]
        url    = urljoin(page_url, href)
        out.append((title, price, rating, url))
    return out

test = get_books(BASE)
len(test), test[0]

(20,
 ('A Light in the Attic',
  'Â£51.77',
  'Three',
  'http://books.toscrape.com/catalogue/a-light-in-the-attic_1000/index.html'))

**Challenge 3 – Seguir paginación (todas las páginas)**

Aquí aprendí que no basta con leer solo la primera página. Muchas webs tienen varias páginas y debemos buscar el botón Next para seguir.
El truco es:

1. Buscar li.next a en el HTML.

2. Usar urljoin para construir el nuevo link.

3. Repetir hasta que no haya más páginas.

In [13]:
def get_all_books(start_url: str, delay=0.5):
    books = []
    url = start_url
    while True:
        r = requests.get(url, headers=headers, timeout=20)
        s = BeautifulSoup(r.text, "lxml")
        # acumular libros de la página actual
        for pod in s.select("article.product_pod"):
            title  = pod.select_one("h3 a")["title"]
            price  = pod.select_one("p.price_color").get_text(strip=True)
            rating = pod.select_one("p.star-rating")["class"][1]
            href   = pod.select_one("h3 a")["href"]
            absurl = urljoin(url, href)
            books.append((title, price, rating, absurl))
        # ¿hay siguiente?
        nxt = s.select_one("li.next a")
        if not nxt:
            break
        url = urljoin(url, nxt["href"])
        time.sleep(delay)
    return books

all_books = get_all_books(BASE)  # ~1000 libros en total
len(all_books), all_books[:3]

(1000,
 [('A Light in the Attic',
   'Â£51.77',
   'Three',
   'http://books.toscrape.com/catalogue/a-light-in-the-attic_1000/index.html'),
  ('Tipping the Velvet',
   'Â£53.74',
   'One',
   'http://books.toscrape.com/catalogue/tipping-the-velvet_999/index.html'),
  ('Soumission',
   'Â£50.10',
   'One',
   'http://books.toscrape.com/catalogue/soumission_998/index.html')])

**Challenge 4 – Extraer por categorías**

Por último, aprendí que también se puede organizar el scraping por categoría, ya que el sitio tiene un menú lateral con enlaces de cada género de libros.

In [14]:
# 1) obtener URLs de categorías
r = requests.get(BASE, headers=headers, timeout=20)
s = BeautifulSoup(r.text, "lxml")
cat_links = [(a.get_text(strip=True), urljoin(BASE, a["href"]))
             for a in s.select("ul.nav-list a") if a.get("href")]
cat_links[:5]

[('Books', 'http://books.toscrape.com/catalogue/category/books_1/index.html'),
 ('Travel',
  'http://books.toscrape.com/catalogue/category/books/travel_2/index.html'),
 ('Mystery',
  'http://books.toscrape.com/catalogue/category/books/mystery_3/index.html'),
 ('Historical Fiction',
  'http://books.toscrape.com/catalogue/category/books/historical-fiction_4/index.html'),
 ('Sequential Art',
  'http://books.toscrape.com/catalogue/category/books/sequential-art_5/index.html')]

In [15]:
# 2) diccionario: categoria -> primeros N libros
def get_books_by_category(limit_per_cat=30, delay=0.5):
    res = {}
    for cat_name, cat_url in cat_links:
        books = get_all_books(cat_url, delay=delay)
        res[cat_name] = books[:limit_per_cat]
        time.sleep(delay)
    return res

buckets = get_books_by_category(limit_per_cat=10, delay=0.2)
list(buckets.keys())[:5], len(buckets)

(['Books', 'Travel', 'Mystery', 'Historical Fiction', 'Sequential Art'], 51)

In [16]:
import pandas as pd
df = pd.DataFrame(all_books, columns=["title","price","rating","url"])
df.head()

# df.to_csv("books.csv", index=False)

,title,price,rating,url
0,A Light in the Attic,Â£51.77,Three,http://books.toscrape.com/catalogue/a-light-in...
1,Tipping the Velvet,Â£53.74,One,http://books.toscrape.com/catalogue/tipping-th...
2,Soumission,Â£50.10,One,http://books.toscrape.com/catalogue/soumission...
3,Sharp Objects,Â£47.82,Four,http://books.toscrape.com/catalogue/sharp-obje...
4,Sapiens: A Brief History of Humankind,Â£54.23,Five,http://books.toscrape.com/catalogue/sapiens-a-...


Conclusión: Dentro de la prectica elegí una pagina para practivar otros escenarios y he logrado aprender que de una página web en HTML plano a una estructura en Python que puedo recorrer y analizar. Con requests obtuve el contenido, con BeautifulSoup lo convertí en algo navegable, y después aprendí a buscar etiquetas, extraer atributos, recorrer páginas y modularizar mi código en funciones.